In [2]:
import math
import requests

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# A descriptive User-Agent is required by Nominatim's usage policy.
HEADERS = {"User-Agent": "nearest-dustbin-finder/1.0 (personal portfolio project)"}


def geocode_address(address: str):
    """
    Convert a free-text address/place name into (lat, lon).
    Returns None if nothing is found.
    """
    params = {"q": address, "format": "json", "limit": 1}
    resp = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    results = resp.json()
    if not results:
        return None
    return float(results[0]["lat"]), float(results[0]["lon"])


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Great-circle distance between two points in meters.
    """
    R = 6371000  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)

    a = (
        math.sin(d_phi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


def find_nearby_bins(lat: float, lon: float, radius_m: int = 1000):
    """
    Query the Overpass API for waste bins, recycling points, and waste
    disposal sites within radius_m meters of (lat, lon).

    Returns a list of dicts: {id, lat, lon, type, name, distance_m}
    sorted by ascending distance.
    """
    # Overpass QL: nodes/ways/relations tagged as waste baskets, recycling
    # points, or waste disposal sites, within a radius around our point.
    # We cast a slightly wider tag net than just waste_basket, since
    # tagging conventions/coverage vary a lot by region.
    query = f"""
    [out:json][timeout:25];
    (
      node["amenity"="waste_basket"](around:{radius_m},{lat},{lon});
      node["amenity"="recycling"](around:{radius_m},{lat},{lon});
      node["amenity"="waste_disposal"](around:{radius_m},{lat},{lon});
      way["amenity"="waste_basket"](around:{radius_m},{lat},{lon});
      way["amenity"="recycling"](around:{radius_m},{lat},{lon});
      way["amenity"="waste_disposal"](around:{radius_m},{lat},{lon});
    );
    out center;
    """
    resp = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    bins = []
    for el in data.get("elements", []):
        if el["type"] == "node":
            b_lat, b_lon = el["lat"], el["lon"]
        elif "center" in el:  # way -> use its centroid
            b_lat, b_lon = el["center"]["lat"], el["center"]["lon"]
        else:
            continue

        tags = el.get("tags", {})
        bin_type = tags.get("amenity", "waste_basket")
        default_names = {
            "recycling": "Recycling point",
            "waste_disposal": "Waste disposal site",
            "waste_basket": "Waste bin",
        }
        name = tags.get("name") or default_names.get(bin_type, "Waste bin")

        dist = haversine_distance(lat, lon, b_lat, b_lon)
        bins.append(
            {
                "id": el["id"],
                "lat": b_lat,
                "lon": b_lon,
                "type": bin_type,
                "name": name,
                "distance_m": round(dist, 1),
                "source": "osm",
            }
        )

    bins.sort(key=lambda b: b["distance_m"])
    return bins


def find_nearby_bins_expanding(lat: float, lon: float, start_radius_m: int = 1000, max_radius_m: int = 8000):
    """
    Like find_nearby_bins(), but if nothing is found at start_radius_m,
    automatically retries at increasing radii (doubling each time) up to
    max_radius_m. This helps in regions where OSM's tagging of individual
    street bins is sparse (common outside major Western cities).

    Returns (bins, radius_used).
    """
    radius = start_radius_m
    bins = find_nearby_bins(lat, lon, radius_m=radius)
    while not bins and radius < max_radius_m:
        radius = min(radius * 2, max_radius_m)
        bins = find_nearby_bins(lat, lon, radius_m=radius)
        if radius == max_radius_m and not bins:
            break
    return bins, radius


def demo_bins(lat: float, lon: float, count: int = 5):
    """
    Generate synthetic demo bin locations scattered near (lat, lon).

    NOT real data — used purely as a fallback so the app remains
    demonstrable in areas where OpenStreetMap has little/no tagged bin
    data (common in many regions outside major Western cities).
    """
    import random

    random.seed(42)  # deterministic layout for a given location
    bins = []
    for i in range(count):
        # scatter within roughly 50-400m
        d_lat = random.uniform(-0.0035, 0.0035)
        d_lon = random.uniform(-0.0035, 0.0035)
        b_lat, b_lon = lat + d_lat, lon + d_lon
        dist = haversine_distance(lat, lon, b_lat, b_lon)
        bins.append(
            {
                "id": f"demo-{i}",
                "lat": b_lat,
                "lon": b_lon,
                "type": "waste_basket",
                "name": f"Demo bin {i + 1}",
                "distance_m": round(dist, 1),
                "source": "demo",
            }
        )
    bins.sort(key=lambda b: b["distance_m"])
    return bins

address = input("Enter an address or place name to search for dustbins:")
location = geocode_address(address)

if location:
    lat, lon = location
    print(f"Geocoded '{address}' to Latitude: {lat}, Longitude: {lon}")
else:
    print(f"Could not geocode '{address}'. Please try again with a more specific address.")

Enter an address or place name to search for dustbins:new york
Geocoded 'new york' to Latitude: 40.7127281, Longitude: -74.0060152


In [3]:
if location:
    bins, radius_used = find_nearby_bins_expanding(lat, lon)
    if bins:
        print(f"Found {len(bins)} bins within {radius_used} meters:")
        for b in bins:
            print(f"  - {b['name']} ({b['type']}) at ({b['lat']:.4f}, {b['lon']:.4f}), Distance: {b['distance_m']}m")
    else:
        print(f"No bins found within {radius_used} meters for '{address}'.")
else:
    print("Cannot search for bins without a valid location.")

Found 733 bins within 1000 meters:
  - Waste bin (waste_basket) at (40.7130, -74.0056), Distance: 46.8m
  - Waste bin (waste_basket) at (40.7122, -74.0059), Distance: 56.7m
  - Waste bin (waste_basket) at (40.7131, -74.0055), Distance: 57.4m
  - Waste bin (waste_basket) at (40.7133, -74.0062), Distance: 66.9m
  - Waste bin (waste_basket) at (40.7130, -74.0053), Distance: 67.5m
  - Waste bin (waste_basket) at (40.7129, -74.0052), Distance: 73.7m
  - Waste bin (waste_basket) at (40.7120, -74.0064), Distance: 86.7m
  - Waste bin (waste_basket) at (40.7129, -74.0050), Distance: 87.3m
  - Waste bin (waste_basket) at (40.7129, -74.0049), Distance: 95.4m
  - Waste bin (waste_basket) at (40.7135, -74.0066), Distance: 100.0m
  - Waste bin (waste_basket) at (40.7134, -74.0068), Distance: 101.3m
  - Waste bin (waste_basket) at (40.7118, -74.0063), Distance: 105.0m
  - Waste bin (waste_basket) at (40.7136, -74.0067), Distance: 109.2m
  - Recycling point (recycling) at (40.7131, -74.0074), Distance

In [4]:
!pip install folium

In [5]:
import folium

if location:
    # Create a map centered at the geocoded location
    m = folium.Map(location=[lat, lon], zoom_start=14)

    # Add a marker for the search location
    folium.Marker(
        location=[lat, lon],
        popup=address,
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)

    # Add markers for each bin found
    if bins:
        for b in bins:
            folium.Marker(
                location=[b['lat'], b['lon']],
                popup=f"{b['name']} ({b['type']}), Distance: {b['distance_m']}m",
                icon=folium.Icon(color='blue', icon='trash')
            ).add_to(m)

    # Display the map
    display(m)
else:
    print("Cannot visualize bins without a valid location.")

In [6]:
import pandas as pd

if 'bins' in locals() and bins:
    df_bins = pd.DataFrame(bins)
    csv_filename = 'nearby_bins.csv'
    df_bins.to_csv(csv_filename, index=False)
    print(f"Successfully exported {len(bins)} bins to '{csv_filename}'")
else:
    print("No bins data available to export. Please ensure bins have been found.")

Successfully exported 733 bins to 'nearby_bins.csv'
